<a href="https://colab.research.google.com/github/muskan-dhawan/DSA-ML-GEN-AI/blob/main/ANN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
%pip install torch

In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ----------------------------
# Load MNIST Dataset
# ----------------------------
transform = transforms.Compose([
    transforms.ToTensor(),                  # Convert image to tensor [0,1]
    transforms.Lambda(lambda x: x.view(-1)) # Flatten 28x28 -> 784
])

train_ds = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_ds = datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

train_loader = DataLoader(
    train_ds,
    batch_size=256,
    shuffle=True,
    num_workers=0
)

test_loader = DataLoader(
    test_ds,
    batch_size=1024,
    shuffle=False,
    num_workers=0
)

# ----------------------------
# Define MLP
# ----------------------------
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(784, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        return self.net(x)

model = MLP().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# ----------------------------
# Training
# ----------------------------
epochs = 5

for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    for xb, yb in train_loader:
        xb = xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad()

        logits = model(xb)
        loss = criterion(logits, yb)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * xb.size(0)

    train_loss = running_loss / len(train_loader.dataset)

    # Evaluation
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for xb, yb in test_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            logits = model(xb)
            preds = logits.argmax(dim=1)

            correct += (preds == yb).sum().item()
            total += yb.size(0)

    test_acc = correct / total

    print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} | Test Accuracy: {test_acc:.4f}")

# ----------------------------
# Final Evaluation
# ----------------------------
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(device)
        yb = yb.to(device)

        preds = model(xb).argmax(dim=1)

        correct += (preds == yb).sum().item()
        total += yb.size(0)

print(f"\nFinal Test Accuracy: {correct / total:.4f}")

Using device: cuda


100%|██████████| 9.91M/9.91M [00:00<00:00, 18.2MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 490kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.01MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 14.4MB/s]


Epoch 1/5 | Train Loss: 0.4482 | Test Accuracy: 0.9403
Epoch 2/5 | Train Loss: 0.1743 | Test Accuracy: 0.9578
Epoch 3/5 | Train Loss: 0.1193 | Test Accuracy: 0.9684
Epoch 4/5 | Train Loss: 0.0882 | Test Accuracy: 0.9731
Epoch 5/5 | Train Loss: 0.0692 | Test Accuracy: 0.9761

Final Test Accuracy: 0.9761
